# 6-3: Visualizations: maps

- mapping with folium and geopandas

## Mapping

- Mapping is a powerful way to visualize data. You can place points on a map, draw lines, and shade areas.

- In this lesson, we will use the `folium` and `geopandas` libraries to create maps.


## Map Visualizations with Folium

To create map visualizations, we will use the Folium, which is a Python wrapper for the Leaflet.js library. 

https://python-visualization.github.io/folium/latest/   

Folium uses https://www.openstreetmap.org/ for drawing maps. Its a free/open source alternative to a service like Google Maps.


In [2]:
import folium
JMA = (43.0362, -76.1363) # geocordinates
map1 = folium.Map(location=JMA, zoom_start=18, tiles="OpenStreetMap", attr="© OpenStreetMap contributors") # centered at JMA
# add a marker, popup text, tooltip on hover. Attach to map object
jma_marker = folium.Marker(JMA, popup='JMA Wireless Dome', tooltip="JMA").add_to(map1)
map1

In [15]:
# Using a different tile set - CartoDB positron

import folium
JMA = (43.0362, -76.1363) # geocordinates
map1 = folium.Map(location=JMA, zoom_start=18, tiles="CartoDB positron", attr="© OpenStreetMap contributors") # centered at JMA
# add a marker, popup text, tooltip on hover. Attach to map object
jma_marker = folium.Marker(JMA, popup='JMA Wireless Dome', tooltip="JMA").add_to(map1)
map1

### Folium and Streamlit

To display folium maps to display in Streamlit, you need to use the `streamlit_folium` module.

This module has two functions: 

- `folium_static` is used to display the map only. Still works but will soon be deprecated. Change `folium_static(map)` to `st_folium(map, returned_objects=[]`) to mimic the static behavior of the old function.
- `st_folium` is use to display the map and return data while the user interacts with the map.

See:
`6-3-st-folium.py` 

### Map Markers Colors and Icons

Through the `icon` named argument you can assign an icon to the popup. You must create a `folium.Icon()` to assign a custom icon. There are three named arguments:

`color=` the color of the marker
`icon_color=` the color of the icon on the marker
`icon=` the name of the icon.


Valid colors are: `['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']`

Valid icons can be found here: https://fontawesome.com/v4/icons/ Your mileage may vary here as not all the icons listed are available.

### Mapping a dataframe

To map a dataframe:

1. we need coordinates
2. we need to use loop over the data, creating a marker for each row



In [9]:
import pandas as pd
classesdf = pd.read_csv("https://raw.githubusercontent.com/mafudge/datasets/master/delimited/class-schedule.csv")
schine = (43.03986, -76.13375)
m = folium.Map(location=schine, zoom_start=17) # create map centered at Schine

for index, row in classesdf.iterrows(): # loop over each course
    text = f"{row['Course']} {row['Day']} {row['Time']} {row['Building']}" # format popup text
    dd = (row['Lat'], row['Lon']) # get lat lon for marker
    if row['Day'] == "TuTh":
        markercolor = 'orange'
    else:
        markercolor = 'darkblue'
    marker = folium.Marker(location=dd, popup=text, icon = folium.Icon(color = markercolor, icon="globe"))
    marker.add_to(m)

m

## Challenge 6-3-1

Streamlit version of Parkmagic!

Load in the `final_cuse_parking_violations.csv` in the `data` folder and create a map of the parking violations in Syracuse.

- drop rows without a latitude and longitude
- provide a dropdown to select the status of the violation
- place pins on the map with the street name and the violation description
- color code the pins based on day of the week 
- to make the map more readable take a random sample of 50 rows from the dataframe



## Geopandas

Geopandas is a library that extends the Pandas library to work with geospatial data. It is built on top of the Shapely, Fiona, and Matplotlib libraries.

https://geopandas.org/

Geopandas can read and write data in many formats, including shapefiles, GeoJSON, and PostGIS. It can also display data on a map.

Geopandas adds a `geometry` column to the dataframe. This column contains the geometry of the object. The geometry can be a POINT, LINE, or POLYGON.

Geopandas has a built in `explore()` function to display the data on a map.

In [16]:
import folium
import geopandas as gpd
schine = (43.03986, -76.13375) # Map Centered at Schine

map = folium.Map(location=schine, zoom_start=17, tiles="OpenStreetMap", attr="© OpenStreetMap contributors")
# Read class schedule data into GeoDataFrame
classesdf = pd.read_csv("https://raw.githubusercontent.com/mafudge/datasets/master/delimited/class-schedule.csv") # Read csv from url. Not a geodataframe yet
# Convert to GeoDataFrame
#points_from_xy() creates Point objects from longitude and latitude columns
# uses Lon → x coordinate and Lat → y coordinate
# Adds a new column called geometry to the dataframe, which contains the Point objects representing the locations of the classes.
gdf = gpd.GeoDataFrame(classesdf, geometry=gpd.points_from_xy(classesdf['Lon'], classesdf['Lat']))
gdf

,Course,Day,Time,Building,Room,Lat,Lon,geometry
0,IST256,M,3:45pm,HBC,Gifford,43.03819,-76.13413,POINT (-76.13413 43.03819)
1,MAT221,TuTh,12:45pm,Bowne,104,43.03674,-76.13320,POINT (-76.1332 43.03674)
2,WRT206,MW,9:20am,Crouse,111,43.03852,-76.13676,POINT (-76.13676 43.03852)
3,COM222,TuTh,9:20am,NH II,232,43.04020,-76.13521,POINT (-76.13521 43.0402)
4,IST343,MW,2:15pm,Hinds,243,43.03834,-76.13364,POINT (-76.13364 43.03834)


In [17]:
map_out = gdf.explore(m=map, marker_type="marker")
map_out

## Choropleth Maps in Geopandas

A choropleth map is a map that uses shading to represent the value of a variable. The shading can be based on a single value or a range of values.

Choropleths maps require a GeoDataFrame with a geometry column. The geometry column contains the shapes that will be displayed on the map.


In [18]:
from matplotlib import colormaps
print(list(colormaps))

['magma', 'inferno', 'plasma', 'viridis', 'cividis', 'twilight', 'twilight_shifted', 'turbo', 'berlin', 'managua', 'vanimo', 'Blues', 'BrBG', 'BuGn', 'BuPu', 'CMRmap', 'GnBu', 'Greens', 'Greys', 'OrRd', 'Oranges', 'PRGn', 'PiYG', 'PuBu', 'PuBuGn', 'PuOr', 'PuRd', 'Purples', 'RdBu', 'RdGy', 'RdPu', 'RdYlBu', 'RdYlGn', 'Reds', 'Spectral', 'Wistia', 'YlGn', 'YlGnBu', 'YlOrBr', 'YlOrRd', 'afmhot', 'autumn', 'binary', 'bone', 'brg', 'bwr', 'cool', 'coolwarm', 'copper', 'cubehelix', 'flag', 'gist_earth', 'gist_gray', 'gist_heat', 'gist_ncar', 'gist_rainbow', 'gist_stern', 'gist_yarg', 'gnuplot', 'gnuplot2', 'gray', 'hot', 'hsv', 'jet', 'nipy_spectral', 'ocean', 'pink', 'prism', 'rainbow', 'seismic', 'spring', 'summer', 'terrain', 'winter', 'Accent', 'Dark2', 'Paired', 'Pastel1', 'Pastel2', 'Set1', 'Set2', 'Set3', 'tab10', 'tab20', 'tab20b', 'tab20c', 'grey', 'gist_grey', 'gist_yerg', 'Grays', 'magma_r', 'inferno_r', 'plasma_r', 'viridis_r', 'cividis_r', 'twilight_r', 'twilight_shifted_r', 't

In [19]:
import random
gdf = gpd.read_file("./data/stlucia.geojson") # read geojson file into GeoDataFrame
gdf['Amount'] = gdf.apply(lambda row: random.randint(50,500), axis=1) # add random Amount column. go over each row and assign random integer between 50 and 500
# You typically do this to create a choropleth map, where the color of each region is based on the value of a specific column (in this case, 'Amount').
gdf

,source,id,name,geometry,Amount
0,https://simplemaps.com,LC06,Gros Islet,"POLYGON ((-60.97997 14.04352, -60.97606 14.061...",302
1,https://simplemaps.com,LC02,Castries,"POLYGON ((-60.97997 14.04352, -60.96559 14.034...",322
2,https://simplemaps.com,LC01,Anse-la-Raye,"POLYGON ((-61.0266 13.98737, -61.00945 13.9817...",473
3,https://simplemaps.com,LC10,Soufrière,"POLYGON ((-60.98063 13.8928, -60.97185 13.8827...",342
4,https://simplemaps.com,LC03,Choiseul,"POLYGON ((-61.0681 13.79218, -61.04956 13.8063...",189
5,https://simplemaps.com,LC07,Laborie,"POLYGON ((-61.01892 13.75625, -61.01203 13.770...",403
6,https://simplemaps.com,LC11,Vieux Fort,"POLYGON ((-60.98158 13.83577, -60.97522 13.839...",316
7,https://simplemaps.com,LC08,Micoud,"POLYGON ((-60.97522 13.83923, -60.96433 13.845...",498
8,https://simplemaps.com,LC09,Praslin,"POLYGON ((-60.96057 13.87901, -60.96057 13.900...",176
9,https://simplemaps.com,LC05,Dennery,"POLYGON ((-60.96057 13.90032, -60.95681 13.927...",316


In [ ]:
#GeoPandas method that creates an interactive web map (using folium) 
gdf.explore(column='Amount', cmap='inferno')

# if prompted trust the notebook ... at the command line run: jupyter trust 6-viz\6-3-slides.ipynb
# This is a security measure to prevent malicious code from running when you open a notebook. 
# By trusting the notebook, you are allowing it to execute any code cells that it contains.


###  Geopandas Choropleth Maps in streamlit.

If you plan to use `GeoDataFrame.explore()` then you will need to call `folium_static()` from the `streamlit_folium` module to display the map in Streamlit.


`6-3-geopandas-px-mapbox.py`



## Challenge 6-3-2

Plot the following data as a choropleth map:

`https://raw.githubusercontent.com/wrobstory/vincent/master/examples/data/US_Unemployment_Oct2012.csv`

1. load the data into pandas
2. load the geometry data into geopandas (in the data folder)
3. merge the data on a common key
4. explore in geopandas!


Issues with streamlit-folium can be addressed here ...
https://github.com/randyzwitch/streamlit-folium/issues